Corrective RAG (CRAG)

Corrective RAG (CRAG) is an advanced technique within Retrieval-Augmented Generation (RAG) that focuses on improving the accuracy and relevance of generated responses by incorporating mechanisms for self-reflection and self-grading of retrieved documents. It does this by evaluating the quality of retrieved documents and applying corrective actions when necessary, such as refining or replacing incorrect retrievals.

Diagram Text
Question
    ↓
Retrieve (Node)
    ↓
Grade (Node)
    ↓
          Any doc irrelevant?
            ◇
         /        \
      No          Yes
      |            |
      |      Re-write query (Node)
      |            |
      |      Web Search (Node)
      |            |
      └────────────┘
            ↓
         Answer
Here's a more detailed explanation:
Addressing Limitations of Basic RAG:

Traditional RAG systems rely heavily on the accuracy of retrieved documents. If the retrieved information is flawed or incomplete, the generated response can also be inaccurate.

CRAG's Core Components:

1. Retrieval Evaluator:
This component assesses the quality and relevance of retrieved documents.

2. Generative Model:
This model generates the initial response based on the retrieved information.

3. Refinement and Correction:
CRAG employs strategies like knowledge refinement or web search to address issues identified by the retrieval evaluator.

Benefits of CRAG:

1. Improved Accuracy:
By evaluating and correcting retrieved information, CRAG helps ensure the accuracy of generated responses.

2. Enhanced Relevance:
CRAG can identify and filter out irrelevant information, making the generated response more relevant.

3. Increased Robustness:
CRAG can handle cases where the initial retrieval process is not perfect, leading to a more robust RAG system.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  ## loading all the environment variable

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

### from langchain_cohere import CohereEmbeddings

# Set embeddings
embd = OpenAIEmbeddings()

# Docs to index
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=0
)

doc_splits = text_splitter.split_documents(docs_list)

# Add to vectorstore
vectorstore = FAISS.from_documents(
    documents=doc_splits,
    embedding=OpenAIEmbeddings()
)

retriever = vectorstore.as_retriever()

In [ ]:
### Retrieval Grader

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

from pydantic import BaseModel, Field


# Data model
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""

    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


# LLM with function call
llm = ChatOpenAI(
    model="gpt-3.5-turbo-0125",
    temperature=0
)

structured_llm_grader = llm.with_structured_output(GradeDocuments)


# Prompt
system = """You are a grader assessing relevance of a retrieved document to a user question. \n
    If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

retrieval_grader = grade_prompt | structured_llm_grader

question = "agent memory"

docs = retriever.invoke(question)

doc_txt = docs[1].page_content

print(
    retrieval_grader.invoke(
        {
            "question": question,
            "document": doc_txt,
        }
    )
)

In [ ]:
### Generate

from langchain import hub
from langchain_core.output_parsers import StrOutputParser

# Prompt
prompt = hub.pull("rlm/rag-prompt")

# LLM
llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    temperature=0
)

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
generation = rag_chain.invoke(
    {
        "context": docs,
        "question": question
    }
)

In [ ]:
### Question Re-writer

# LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo-0125",
    temperature=0
)

# Prompt
system = """You a question re-writer that converts an input question to a better version that is optimized \n
    for web search. Look at the input and try to reason about the underlying semantic intent / meaning."""

re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        (
            "human",
            "Here is the initial question: \n\n {question} \n Formulate an improved question.",
        ),
    ]
)

question_rewriter = re_write_prompt | llm | StrOutputParser()

question_rewriter.invoke({"question": question})

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(k=3)

In [ ]:

from typing import TypedDict
from typing import List

class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        question: question
        generation: LLM generation
        web_search: whether to add search
        documents: list of documents
    """

    question: str
    generation: str
    web_search: str
    documents: List[str]

In [ ]:
def retrieve(state):
    """
    Retrieve documents

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, documents, that contains retrieved documents
    """
    print("---RETRIEVE---")

    question = state["question"]

    # Retrieval
    documents = retriever.invoke(question)

    return {
        "documents": documents,
        "question": question,
    }

